# kx: one-line matplotlib plots and themes

[`kx`](https://github.com/vhrabar/kx-plot) is a small plain-Python package with `.mplstyle` themes, palettes and colourmaps. Install it with `pip install kx-plot`, or attach the [Kaggle dataset](https://www.kaggle.com/datasets/vhrabar/kx-plot) to use it **offline** (no internet), so it also runs in competition notebooks.

```python
kx.plot(x, y, z, 'scatter-dark-grid')   # spec = kind-theme-flag-flag-...
```

**Contents**
1. Setup
2. Discovery
3. Kinds
4. Call shapes
5. Flags
6. Themes
7. Palettes, colormaps and rcParams
8. Subplots with `ax=`
9. Saving figures
10. Strict errors
11. Transparency

## 1. Setup

Attach the dataset: **Add Input → Datasets → `vhrabar/kx-plot`**. It mounts read-only at the path below; put that folder on `sys.path` and import `kx`.

`kx.use(theme)` sets the notebook-wide default theme.

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/vhrabar/kx-plot')

import kx
kx.use('dark')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Discovery

Everything valid is listable: kinds, themes and flags with `kx.grammar()` (or `kx.grammar('kde')` for one kind), themes with `kx.styles()`, bundled demo data with `kx.datasets()`.

In [ ]:
kx.grammar()

In [ ]:
kx.grammar('hist')

In [ ]:
print('themes  :', kx.styles())
print('datasets:', kx.datasets())

`kx.load(name)` reads one of the bundled CSVs into a DataFrame.

In [ ]:
waves   = kx.load('waves')     # x, y, z: two smooth curves
cloud   = kx.load('cloud')     # x, y, z: correlated point cloud
monthly = kx.load('monthly')   # month, y2025, y2026: counts per month
dists   = kx.load('dists')     # normal, skewed, bimodal: samples
daily   = kx.load('daily')     # date, temp, sales: one row per day, 2025-2026
monthly.head()

## 3. Kinds

| kind      | `x`             | `y`           | `z` (optional unless noted)                                                               |
|-----------|-----------------|---------------|-------------------------------------------------------------------------------------------|
| `line`    | x values        | first line    | second line                                                                               |
| `step`    | x values        | first line    | second line, both drawn as steps                                                          |
| `area`    | x values        | first area    | second area; flags `stack` and `norm`                                                     |
| `band`    | x values        | centre line   | **required**: half-width, shaded `y - z` to `y + z`                                       |
| `scatter` | x values        | y values      | numbers → colour + colorbar; text, bool or categorical → one colour per category + legend |
| `bar`     | category labels | bar heights   | second group, side by side or stacked                                                     |
| `barh`    | category labels | bar lengths   | like `bar`, lying down, first category on top                                             |
| `hist`    | first sample    | second sample | third sample, all on the same bins                                                        |
| `kde`     | first sample    | second sample | third sample, one smooth density curve each                                               |
| `ecdf`    | first sample    | second sample | third sample, one step curve each                                                         |
| `box`     | first sample    | second sample | third sample, boxes side by side                                                          |
| `violin`  | first sample    | second sample | third sample, violins side by side                                                        |

For the distribution kinds (`hist`, `kde`, `ecdf`, `box`, `violin`) `y` and `z` are optional, and `kde`, `ecdf`, `box` and `violin` can split one sample into groups with `by=` (section 4).

Without a kind in the spec, `line` is used.

### `line`

In [ ]:
kx.plot(waves.x, waves.y, 'line', title='line: one series');

In [ ]:
kx.plot(waves.x, waves.y, waves.z, 'line-leg', title='line: y and z against x');

### `step`

Like `line`, drawn as steps centred on each x. Good for counts per period.

In [ ]:
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'step-leg-grid', title='step: monthly counts');

### `area`

| spec         | result                                                                  |
|--------------|-------------------------------------------------------------------------|
| `area`       | each series filled down to 0, semi-transparent so overlaps stay visible |
| `area-stack` | `z` stacked on top of `y`; the top edge is the total                    |
| `area-norm`  | 100% stack: each series' share of the total at every x (needs `z`)      |

In [ ]:
kx.plot(waves.x, waves.y + 1.2, waves.z + 1.2, 'area-leg', title='area: overlapping');

In [ ]:
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'area-stack-leg', title='area-stack: total per month');

In [ ]:
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'area-norm-leg', title='area-norm: share per month');

With `norm` the area fills the whole plot, so `leg` puts the legend outside on the right.

### `band`

A centre line `y` with a shaded band from `y - z` to `y + z` in the same colour, e.g. mean ± std or a forecast ± error. `z` can be a single number repeated, or vary along x.

In [ ]:
t = waves.x
spread = pd.Series(0.1 + t / 40, name='std')           # uncertainty growing with x
kx.plot(t, waves.y, spread, 'band-leg-grid', title='band: y ± std');

### `scatter`

What `z` does depends on its type:
- **numbers** colour the points and add a colorbar (colormap from `kx.use(cmap=...)`, see section 7)
- **text, bool or pandas categorical** gives one colour per category and a legend titled with `z`'s name. Categories are sorted, unless `z` is a `pd.Categorical`, which keeps its own order.

In [ ]:
kx.plot(cloud.x, cloud.y, 'scatter', title='scatter: plain');

In [ ]:
kx.plot(cloud.x, cloud.y, cloud.z, 'scatter-grid', title='scatter: numeric z → colorbar');

In [ ]:
level = pd.Series(pd.cut(cloud.z, [-np.inf, -1, 1, np.inf], labels=['low', 'mid', 'high']), name='z level')
kx.plot(cloud.x, cloud.y, level, 'scatter-grid', title='scatter: categorical z → legend');

In [ ]:
kx.plot(cloud.x, cloud.y, (cloud.z > 0).rename('z > 0'), 'scatter', title='scatter: bool z → two groups');

### `bar`

`x` gives the category labels; with `z` the bars are grouped side by side, or with `stack` `z` is drawn on top of `y` so the bar's top is the total.

In [ ]:
kx.plot(monthly.month, monthly.y2025, 'bar', title='bar: one group');

In [ ]:
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'bar-leg', title='bar: grouped');

In [ ]:
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'bar-stack-leg', title='bar-stack: total per month');

### `barh`

Like `bar`, lying down: the first category is on top, so it reads like a table. Long category names fit better this way. `stack` works the same.

In [ ]:
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'barh-leg', title='barh: grouped');

In [ ]:
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'barh-stack-leg', title='barh-stack');

### `hist`

Up to three samples on the same 30 bins, so overlaid bars line up. `y` is optional here.

| spec            | result                                                                    |
|-----------------|---------------------------------------------------------------------------|
| `hist`          | overlaid, semi-transparent                                                |
| `hist-stack`    | stacked on top of each other                                              |
| `hist-norm`     | density: each histogram has area 1, so samples of different sizes compare |
| `hist-cum`      | cumulative counts, drawn as outlines so they don't hide each other        |
| `hist-norm-cum` | the CDF, from 0 to 1                                                      |

In [ ]:
kx.plot(dists.normal, spec='hist', title='hist: one sample');

In [ ]:
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'hist-leg', title='hist: three samples');

In [ ]:
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'hist-stack-leg', title='hist-stack');

In [ ]:
kx.plot(dists.normal, dists.skewed[:200], 'hist-norm-leg', title='hist-norm: 200 vs all values, same scale');

In [ ]:
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'hist-norm-cum-leg-grid', title='hist-norm-cum: the CDF');

### `kde`

Smooth density curves, one per sample: a Gaussian kernel density estimate with Silverman's bandwidth, computed in numpy (no scipy needed). Each curve has area 1.

In [ ]:
kx.plot(dists.normal, spec='kde', title='kde: one sample');

In [ ]:
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'kde-leg', title='kde: three samples');

### `ecdf`

The empirical CDF: at each x, the share of values at or below it. Nothing is binned or smoothed, so every value counts. Reading off a quantile is easy: where a curve crosses 0.5 is its median.

In [ ]:
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'ecdf-leg-grid', title='ecdf');

### `box`

Boxes side by side: the median, the quartile box, whiskers to 1.5 × IQR and the points beyond as outliers. Each box is named after its sample.

In [ ]:
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'box-grid', title='box');

### `violin`

A mirrored density per sample, with the quartiles as a thick line and the median as a dot. Unlike `box`, it shows that `bimodal` has two peaks.

In [ ]:
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'violin-grid', title='violin');

## 4. Call shapes

- `kx.plot(x, y, z, spec)` is the full form.
- `kx.plot(df, 'x', 'y', 'z', spec)` names columns of a DataFrame instead; `by=` can name a column too.
- With no `z`, the spec can go third: `kx.plot(x, y, 'bar')`.
- `spec=`, `title=` and `ax=` also work as keywords.
- Any other keyword goes to the matplotlib call that draws the kind and wins over kx's own defaults (below).
- `by=` gives a group per `x` value for `kde`, `ecdf`, `box` and `violin` (below).
- A pandas Series' name becomes its legend label; plain numpy arrays are labelled `y` and `z`.

`kx.plot` returns the matplotlib `Axes`, so you can keep customising it.

In [ ]:
rng = np.random.default_rng(0)
t = np.linspace(0, 4 * np.pi, 200)
a, b = np.sin(t), np.sin(t) * np.exp(-t / 8)

ax = kx.plot(t, a, b, spec='line-leg', title='numpy arrays: labels y and z')
ax.set_xlabel('t (rad)'); ax.set_ylabel('amplitude');

In [ ]:
df = pd.DataFrame({'t': t, 'signal': a, 'damped': b})
kx.plot(df.t, df.signal, df.damped, 'line-leg', title='Series: column names as labels');

With a DataFrame first, the next strings are column names, and a last string that isn't a column is the spec. A column named like a kind (say `bar`) is still a column. A mistyped column gets the closest names as a hint.

In [ ]:
kx.plot(df, 't', 'signal', 'damped', 'line-leg', title='DataFrame: columns by name');
kx.plot(monthly, 'month', 'y2025', 'y2026', 'bar-leg', title='the same call shape for any kind');

In [ ]:
try:
    kx.plot(df, 't', 'singal')
except ValueError as e:
    print('ValueError :', e)

### Matplotlib keywords

Keywords `kx.plot` doesn't know are passed to the matplotlib call that draws the kind, and they win over kx's defaults: `ax.plot` for `line` and `kde`, `ax.scatter` for `scatter`, `ax.bar` for `bar`, the fills for `area`, `ax.boxplot` for `box` and so on (`kx.grammar(kind)` says which). Two are read by kx itself:
- `bins=` for `hist`: a count, the edges, or a numpy rule like `'auto'`, still shared by every series
- `width=` for `bar` (`height=` for `barh`): the space each category's bars fill, 0.8 by default

In [ ]:
kx.plot(waves.x, waves.y, waves.z, 'line-leg', linewidth=3, linestyle='--', title='linewidth=3, linestyle=--');
kx.plot(cloud.x, cloud.y, cloud.z, 'scatter', cmap='magma', s=40, alpha=0.6, title='cmap, s, alpha for this plot only');

In [ ]:
kx.plot(dists.normal, dists.skewed, spec='hist-leg', bins=60, title='bins=60');
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'bar-leg', width=0.5, edgecolor='black', title='width=0.5, edgecolor');
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'box', showfliers=False, title='box, showfliers=False');

The spec can't be given twice (third argument *and* `spec=`), and every kind except the distribution kinds (`hist`, `kde`, `ecdf`, `box`, `violin`) needs `y`.

In [ ]:
for call in (lambda: kx.plot(t, a, 'line', spec='bar'),
             lambda: kx.plot(t, spec='line')):
    try:
        call()
    except (TypeError, ValueError) as e:
        print(type(e).__name__, ':', e)

### Groups with `by=`

Data often comes long: one column of values and one column saying which group each value is in. `by=` splits `x` into one curve, box or violin per group, and names the legend or the axis after the `by` Series. Groups are sorted, unless `by` is a `pd.Categorical`, which keeps its own order.

In [ ]:
long = dists.melt(var_name='dist', value_name='value')   # 3 columns -> value + group
long.sample(5, random_state=0)

In [ ]:
kx.plot(long, 'value', spec='kde', by='dist', title='kde, by=dist');
kx.plot(long.value, spec='ecdf-grid', by=long.dist, title='ecdf, by=dist');

In [ ]:
order = pd.Categorical(long.dist, categories=['bimodal', 'skewed', 'normal'])
kx.plot(long.value, spec='box', by=pd.Series(order, name='dist'), title='box, categorical order');
kx.plot(long.value, spec='violin', by=long.dist, title='violin, by=dist');

`by=` only works with those four kinds, and replaces `y` and `z`:

In [ ]:
for call in (lambda: kx.plot(long.value, spec='hist', by=long.dist),
             lambda: kx.plot(long.value, long.value, spec='kde', by=long.dist)):
    try:
        call()
    except ValueError as e:
        print('ValueError :', e)
plt.close('all')

## 5. Flags

Flags that work with every kind:

| flag                      | effect                                                           |
|---------------------------|------------------------------------------------------------------|
| `grid`                    | turn on the grid                                                 |
| `leg`                     | show the legend                                                  |
| `legout`                  | legend outside, right of the plot                                |
| `tight`                   | `tight_layout()` on the figure                                   |
| `rot`                     | x tick labels rotated 45°                                        |
| `logx` / `logy` / `logxy` | log scale on x, y or both                                        |
| `symlogy`                 | symmetric log y: log both ways, linear near 0, so negatives work |
| `zero`                    | stretch the value axis to include 0                              |
| `eq`                      | equal aspect: one unit is as long on x as on y                   |
| `pct`                     | value axis as percent: 0.25 → 25%                                |
| `si`                      | value axis with SI prefixes: 1500 → 1.5k                         |

The value axis is y, except for `barh`, where it is x.

Flags for some kinds:

| flag              | kinds                                     | effect                                                        |
|-------------------|-------------------------------------------|---------------------------------------------------------------|
| `stack`           | `area`, `bar`, `barh`, `hist`             | stack the series                                              |
| `norm`            | `area`, `hist`                            | `area`: 100% stack (implies `stack`); `hist`: density, area 1 |
| `cum`             | `hist`                                    | cumulative; `hist-norm-cum` is the CDF                        |
| `date`            | `line`, `step`, `area`, `band`, `scatter` | `x` read as dates (strings work too), compact date ticks      |
| `mk`              | `line`, `step`, `area`, `band`            | a marker on every point                                       |
| `sort`            | `bar`, `barh`                             | largest first (by `y`, or by the total with `stack`)          |
| `ann`             | `bar`, `barh`                             | the value on each bar                                         |
| `mean` / `median` | `hist`, `kde`, `ecdf`                     | a dashed / dotted line at each series' mean / median          |
| `mean`            | `box`, `violin`                           | a diamond at each mean                                        |
| `ann`             | `hist`, `kde`, `ecdf`, `box`, `violin`    | write the mean / median values                                |
| `nocb`            | `scatter` with numeric `z`                | no colorbar                                                   |
| `sym`             | `scatter` with numeric `z`                | colour limits ±max\|z\|, so 0 is the middle colour            |
| `logc`            | `scatter` with numeric `z`                | log colour scale (`z` > 0)                                    |

Tokens can come in any order, so `line-grid-leg` and `leg-grid-line` are the same plot.

In [ ]:
x = np.linspace(1, 100, 200)
kx.plot(x, x ** 2, x ** 3, 'line-logy-grid-leg', title='logy: x² and x³ span six decades');

In [ ]:
kx.plot(x, np.log(x), 'logx-grid-line', title='logx: log(x) becomes a straight line');

In [ ]:
kx.plot(dists.skewed, spec='hist-logy-tight', title='hist + logy: the thin tail stays visible');

### Layout: `legout`, `rot`

In [ ]:
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'line-legout-rot-grid', title='legout + rot');

### Axes: `logxy`, `symlogy`, `zero`, `eq`, `date`

In [ ]:
kx.plot(x, 3 * x ** 1.5, 'line-logxy-grid', title='logxy: a power law is a straight line');

In [ ]:
s = np.linspace(-6, 6, 200)
kx.plot(s, np.sinh(s), 'line-symlogy-grid', title='symlogy: sinh grows both ways');

In [ ]:
kx.plot(monthly.month, monthly.y2026, 'line-mk', title='without zero: the dip looks deep');
kx.plot(monthly.month, monthly.y2026, 'line-mk-zero', title='zero: the same dip, to scale');

In [ ]:
kx.plot(cloud.x, cloud.y, 'scatter-eq-grid', title='eq: one unit is as long on both axes');

In [ ]:
kx.plot(daily.date, daily.temp, 'line-date-grid', title='date: text dates, compact ticks');
kx.plot(daily.date[:56], daily.sales[:56], 'line-date-mk-zero', title='date-mk: eight weeks of sales');

### Number format: `pct`, `si`

In [ ]:
share = monthly.y2026 / (monthly.y2025 + monthly.y2026)
kx.plot(monthly.month, share.rename('2026 share'), 'bar-pct-rot', title='pct: 0.5 → 50%');
kx.plot(monthly.month, monthly.y2025 * 1000, 'barh-si-sort', title='si: 145000 → 145k');

### Markers and annotations: `mk`, `ann`, `mean`, `median`, `sort`

`ann` on bars writes each value; with `stack` it writes each segment's value inside it.

In [ ]:
kx.plot(monthly.month, monthly.y2025, 'barh-sort-ann', title='barh-sort-ann: a ranked table');
kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'bar-stack-sort-ann-leg-rot', title='bar-stack-sort-ann');

In [ ]:
kx.plot(dists.skewed, spec='hist-mean-median-ann', title='hist: mean (dashed) is pulled into the tail');
kx.plot(dists.normal, dists.bimodal, spec='ecdf-median-ann-pct-leg', title='ecdf: the median is where 50% is crossed');

In [ ]:
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'box-mean-ann', title='box: median and mean (μ) per box');

### Colour scale: `nocb`, `sym`, `logc`

In [ ]:
kx.plot(cloud.x, cloud.y, cloud.z, 'scatter-sym', title='sym: 0 is the middle colour');
kx.plot(cloud.x, cloud.y, np.exp(cloud.z).rename('exp z'), 'scatter-logc', title='logc: values over 3 decades');
kx.plot(cloud.x, cloud.y, cloud.z, 'scatter-nocb', title='nocb: colour, no colorbar');

## 6. Themes

| theme     | for                | what it sets                                                               |
|-----------|--------------------|----------------------------------------------------------------------------|
| `dark`    | notebooks          | dark background, muted colours                                             |
| `light`   | notebooks          | white background, tab10-style colours                                      |
| `paper`   | greyscale print    | black-to-grey colours paired with a linestyle cycle, serif, 300 dpi export |
| `science` | one journal column | 3.5 in wide, 8 pt serif, inward ticks on all sides, 600 dpi export         |
| `talk`    | slides             | 16:9, 18 pt sans, thick lines                                              |
| `poster`  | posters            | 12×9 in, 26 pt sans, very thick lines                                      |
| `thesis`  | A4 thesis          | 160 mm text width, Computer Modern 10/9 pt (no LaTeX needed), PDF export   |

There are two ways to use a theme:
- **`kx.use('paper')`** sets the default for the rest of the notebook. It resets matplotlib first, so themes never mix.
- **A theme token in the spec**, as in `'line-paper'`, styles that one plot and leaves the notebook default alone.

In [ ]:
kx.plot(waves.x, waves.y, waves.z, 'line-paper-leg', title='paper, just this plot');
kx.plot(waves.x, waves.y, waves.z, 'line-leg', title='still dark');

Each theme at its own figure size:

In [ ]:
for theme in kx.styles():
    ax = kx.plot(waves.x, waves.y, waves.z, f'line-{theme}-leg', title=theme)
    ax.set_xlabel('time (s)'); ax.set_ylabel('value')
    plt.show()

Every kind in one theme, by passing `ax=` (see section 8):

In [ ]:
def overview(theme: str) -> None:
    kx.use(theme)
    w, h = plt.rcParams['figure.figsize']
    fig, axs = plt.subplots(3, 4, figsize=(4 * w, 3 * h))
    a = iter(axs.flat)
    samples = (dists.normal, dists.skewed, dists.bimodal)
    kx.plot(waves.x, waves.y, waves.z, 'line-leg', title='line', ax=next(a))
    kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'step-leg', title='step', ax=next(a))
    kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'area-stack-leg', title='area-stack', ax=next(a))
    kx.plot(waves.x, waves.y, (0.1 + waves.x / 40).rename('std'), 'band-leg', title='band', ax=next(a))
    kx.plot(cloud.x, cloud.y, cloud.z, 'scatter', title='scatter', ax=next(a))
    kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'bar-leg', title='bar', ax=next(a))
    kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'barh-stack-leg', title='barh-stack', ax=next(a))
    for k in ('hist', 'kde', 'ecdf', 'box', 'violin'):
        kx.plot(*samples, spec=k + ('-leg' if k in ('hist', 'kde', 'ecdf') else ''), title=k, ax=next(a))
    fig.suptitle(theme); fig.tight_layout(); plt.show()

overview('science')
overview('paper')
kx.use('dark')

## 7. Palettes, colormaps and rcParams

### Palettes

`kx.use(theme, palette=...)` swaps the theme's colours for a **named palette** from `kx/styles/palettes/` (`kx.palettes()` lists them, `kx.palette(name)` returns the colours) or for your own list of colours.

| palettes                                            | source                                                                                     |
|-----------------------------------------------------|--------------------------------------------------------------------------------------------|
| `okabe`                                             | Okabe & Ito: 8 colours that stay distinguishable for every common type of colour blindness |
| `tolbright` `tolvibrant` `tollight` `tolhc` `tolmc` | Paul Tol                                                                                   |
| `petroff6` `petroff8` `petroff10`                   | Petroff (2021)                                                                             |
| `tab10` `tab20` `tableaucb`                         | Tableau; `tab20` has 20 colours, light/dark pairs                                          |
| `set1` `set2` `dark2` `paired`                      | ColorBrewer; `paired` has 12, light/dark pairs                                             |
| `ibm`                                               | IBM Design                                                                                 |

In [ ]:
print(kx.palettes())
kx.palette('okabe')

In [ ]:
names = kx.palettes()
fig, ax = plt.subplots(figsize=(10, 0.4 * len(names)))
for row, name in enumerate(names):
    for col, c in enumerate(kx.palette(name)):
        ax.add_patch(plt.Rectangle((col, row), 0.9, 0.8, color=c))
ax.set_xlim(0, 20); ax.set_ylim(len(names), -0.2)
ax.set_yticks([r + 0.4 for r in range(len(names))], names); ax.set_xticks([])
ax.set_title('every palette');

In [ ]:
kx.use('light', palette='okabe')
fig, ax = plt.subplots(figsize=(8, 3.5))
for i, c in enumerate(kx.palette('okabe')):
    ax.plot(waves.x, waves.y + i, label=c)
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.12)); ax.set_title('okabe on light');

Only the colours change. Other properties the theme cycles through stay, such as `paper`'s linestyles, so `paper` + `okabe` gives colour for screens and dash patterns for greyscale print.

In [ ]:
kx.use('paper', palette='okabe')
kx.plot(waves.x, waves.y, waves.z, 'line-leg', title='paper + okabe');

### Colormaps

`kx.use(theme, cmap=...)` sets the colormap used for continuous colour, such as `scatter` with `z`. `kx.cmaps()` lists the supported ones that your matplotlib version has (`berlin`, `managua` and `vanimo` need matplotlib 3.10 or newer). The cmocean and Crameri ones are vendored in `kx/styles/cmaps/`, so they work with any matplotlib.

| type                   | cmaps                                                  | use for                                                                                       |
|------------------------|--------------------------------------------------------|-----------------------------------------------------------------------------------------------|
| sequential             | `viridis` `magma` `plasma` `inferno` `cividis` `turbo` | low → high values (`cividis` is colour-blind safe; `turbo` is a rainbow, for spotting detail) |
| sequential, cmocean    | `thermal` `haline` `deep`                              | temperature, salinity, depth                                                                  |
| sequential, Crameri    | `batlow` `lipari` `hawaii` `oslo` `davos`              | low → high, perceptually uniform and colour-blind friendly                                    |
| cyclic                 | `twilight` `romaO`                                     | angles, phase, time of day                                                                    |
| diverging              | `coolwarm` `balance` `vik` `roma` `bam` `cork`         | values around a midpoint such as 0                                                            |
| diverging, dark centre | `berlin` `managua` `vanimo`                            | as above, on light backgrounds                                                                |

`berlin`, `managua` and `vanimo` are dark in the middle, so on the `dark` theme values near 0 fade into the background. Use `coolwarm` or `vik` there, or pair the dark-centred ones with `light`.

Add `_r` to any name to reverse it: `cmap='batlow_r'`.

In [ ]:
import matplotlib
print('matplotlib', matplotlib.__version__)
kx.cmaps()

`cmap=` just sets matplotlib's `image.cmap`, so for one-off panels inside a figure `plt.rc_context` does the same job:

In [ ]:
kx.use('dark')
names = kx.cmaps()
cols = 5
rows = -(-len(names) // cols)
fig, axs = plt.subplots(rows, cols, figsize=(3.4 * cols, 3 * rows))
for ax, name in zip(axs.flat, names):
    with plt.rc_context({'image.cmap': name}):
        kx.plot(cloud.x, cloud.y, cloud.z, 'scatter', title=name, ax=ax)
for ax in axs.flat[len(names):]:
    ax.set_visible(False)
fig.tight_layout()

In [ ]:
kx.use('dark', cmap='batlow_r')                    # '_r' reverses any colormap in kx.cmaps()
kx.plot(cloud.x, cloud.y, cloud.z, 'scatter-grid', title='batlow_r');

In [ ]:
kx.use('light', palette='tollight', cmap='vik')   # theme, palette and colormap together
kx.plot(cloud.x, cloud.y, cloud.z, 'scatter-grid', title='light + tollight + vik');
kx.plot(dists.normal, dists.skewed, dists.bimodal, 'kde-leg', title='light + tollight');

### rcParams

Any rcParam can also be passed as a keyword. Write the dots in rcParam names as double underscores: `figure__figsize` means `figure.figsize`.

In [ ]:
kx.use('light', palette=['#264653', '#2a9d8f', '#e76f51'],   # or your own colours
       figure__figsize=(7, 3), lines__linewidth=3, axes__grid=True)
kx.plot(waves.x, waves.y, waves.z, 'line-leg', title='light + custom palette and rcParams');
kx.use('dark')

## 8. Subplots with `ax=`

Pass `ax=` to draw into your own layout. `kx.plot` returns that axis.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
kx.plot(cloud.x, cloud.y, cloud.z, 'scatter-grid', title='features', ax=left)
kx.plot(cloud.z, spec='hist', title='z distribution', ax=right)
fig.tight_layout()

## 9. Saving figures

Kaggle only keeps files written to `/kaggle/working`. The print themes come with sensible save settings: `paper` saves at 300 dpi, `science` at 600 dpi, and `thesis` saves a vector PDF when the file name has no extension. Use `with plt.style.context(...)` or `kx.use` so those settings are active when you save.

In [ ]:
import os
out = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

kx.use('science')
ax = kx.plot(waves.x, waves.y, waves.z, 'line-leg')
ax.set_xlabel('time (s)'); ax.set_ylabel('signal')
ax.figure.savefig(f'{out}/figure_science.png')   # 600 dpi, tight bbox, white background
plt.show()

kx.use('thesis')
ax = kx.plot(monthly.month, monthly.y2025, monthly.y2026, 'bar-leg')
ax.figure.savefig(f'{out}/figure_thesis')          # no extension -> PDF
plt.show()

kx.use('dark')
sorted(f for f in os.listdir(out) if f.startswith('figure_'))

## 10. Strict errors

Unknown tokens, two kinds, two themes, a flag on a kind that doesn't support it (such as `cum` outside `hist`), flags that ask for opposite things (such as `pct` and `si`), an unknown colormap, or a missing required `z` raise a `ValueError` instead of being guessed at.

In [ ]:
for spec in ('line-nope', 'line--dark', 'line-scatter', 'dark-light', 'line-stack', 'line-cum', 'kde-stack', 'band', 'area-norm',
             'line-pct-si', 'line-logy-logxy', 'line-sort', 'hist-ann', 'scatter-nocb'):
    try:
        kx.plot(waves.x, waves.y, spec=spec)
    except ValueError as e:
        print(f'{spec!r:18} {e}')
try:
    kx.use('dark', cmap='rainbow')
except ValueError as e:
    print(f"{'cmap=rainbow':18} {e}")
plt.close('all')

## 11. Transparency

Nothing is hidden. `kx.show(theme)` prints a theme file, `kx.source(kind)` prints the code that draws one kind and `kx.source()` prints every file in the package.

In [ ]:
kx.show('science')

In [ ]:
kx.source('violin')

In [ ]:
kx.source()

---

Source, issues and the feature tracker: [github.com/vhrabar/kx-plot](https://github.com/vhrabar/kx-plot)